In [2]:
import os
import shutil
import pandas as pd
from tqdm import tqdm
import json

In [12]:
files_status = pd.read_excel('files_status.xlsx').fillna('').to_dict(orient='records')
files_status = {e['filename']:e['status'] + e['note'].replace('wchodzi', '') for e in files_status}

In [ ]:
for file in tqdm(os.listdir('manual_to_prepare')):
    filename = file.replace('.xlsx', '')
    if files_status.get(filename) == 'manualne':
        src = 'manual_to_prepare/' + file
        dst = 'manual/' + file
        shutil.copy(src, dst)

In [ ]:
# filter automatic full
for file, status in tqdm(files_status.items()):
    if status == 'automatyczne':
        try:
            src = 'remote_classification_output/filtered_files/filtered_' + file + '.json'
            dst = 'to_bibframe/automatic_filtered/' + file + '.json'
            shutil.copy(src, dst)
        except: print(file)

In [51]:
# biuro literackie dodatek automatyczny
biuro_df = pd.read_excel('to_bibframe/manual/biuroliterackie_2022-11-08.xlsx', sheet_name='Posts').fillna('')
biuro_df = biuro_df[(biuro_df['do PBL'] != True) 
                    & (biuro_df['Sekcja'] == 'Biblioteka') 
                    & biuro_df['Kategoria'].str.startswith(('debaty', 'książki', 'recenzje', 'utwory', 'wywiady'))]

biuro_urls = set(biuro_df['Link'].to_list())

with open('all/biuroliterackie_2022-11-08.json', 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)

new_biuro_literackie = [e for e in data if e['Link'] in biuro_urls and e['Sekcja'] == 'biblioteka']
new_biuro_literackie = {e['Link']:e for e in new_biuro_literackie}
new_biuro_literackie = [v for k, v in new_biuro_literackie.items()]

with open('to_bibframe/automatic_filtered/biuroliterackie_2022-11-08_dodatek.json', 'w', encoding='utf-8') as jfile:
        json.dump(new_biuro_literackie, jfile, ensure_ascii=False)

In [7]:
# pisarze.pl dodatek automatyczny
pisarze_df = pd.read_excel('to_bibframe/manual/pisarze_2023-01-27.xlsx', sheet_name='Posts').fillna('')
pisarze_df = pisarze_df[(pisarze_df['do PBL'] != True)
                    & pisarze_df['Kategoria'].str.contains('Wiersze|Wiersz dnia|Poezja|Proza', na=False)]


pisarze_urls = set(pisarze_df['Link'].to_list())

with open('all/pisarze_2023-01-27.json', 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)

new_pisarze_pl = [e for e in data if e['Link'] in pisarze_urls]
new_pisarze_pl = {e['Link']:e for e in new_pisarze_pl}
new_pisarze_pl = [v for k, v in new_pisarze_pl.items()]

with open('to_bibframe/automatic_filtered/pisarze_2023-01-27_dodatek.json', 'w', encoding='utf-8') as jfile:
        json.dump(new_pisarze_pl, jfile, ensure_ascii=False)

In [23]:
# dwutygodnik dodatek automatyczny
dwutygodnik_df = pd.read_excel('to_bibframe/manual/dwutygodnik_2024-05-06.xlsx', sheet_name='Posts').fillna('')
dwutygodnik_df = dwutygodnik_df[(dwutygodnik_df['do PBL'] != True) & (dwutygodnik_df['Sekcja'] == 'Felietony')]

dwutygodnik_urls = set(dwutygodnik_df['Link'].to_list())

with open('all/dwutygodnik_2024-05-06.json', 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)

new_dwutygodnik = [e for e in data if e['Link'] in dwutygodnik_urls]
new_dwutygodnik = {e['Link']:e for e in new_dwutygodnik}
new_dwutygodnik = [v for k, v in new_dwutygodnik.items()]

with open('to_bibframe/automatic_filtered/dwutygodnik_2024-05-06_dodatek.json', 'w', encoding='utf-8') as jfile:
        json.dump(new_dwutygodnik, jfile, ensure_ascii=False)

In [ ]:
# count records
counter_manual = 0
for file in tqdm(os.listdir('to_bibframe/manual')):
    df = pd.read_excel('to_bibframe/manual/' + file, sheet_name='Posts').fillna('')
    df = df[df['do PBL'] == True]
    counter_manual += len(df)
print('counter_manual', counter_manual)

counter_automatic_full = 0
for file in tqdm(os.listdir('to_bibframe/automatic_full')):
    with open('to_bibframe/automatic_full/' + file, 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)
        counter_automatic_full += len(data)

print('counter_automatic_full', counter_automatic_full)

counter_automatic_filtered = 0
for file in tqdm(os.listdir('to_bibframe/automatic_filtered')):
    with open('to_bibframe/automatic_filtered/' + file, 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)
        counter_automatic_filtered += len(data)

print('counter_automatic_filtered', counter_automatic_filtered)

print('all records: ', counter_manual + counter_automatic_full + counter_automatic_filtered)

In [ ]:
# classification_results_sztukater_2026-05-04.json

with open('remote_classification_output/classification_results/classification_results_sztukater_2026-05-04.json', 'r', encoding='utf-8') as jfile:
        data = json.load(jfile) 
print(len([e for e in data]))
print(len([e for e in data if e[1]]))

In [82]:
df = pd.read_excel('Internet Archive status.xlsx')
df = df[~df['internet_archive_url'].isna()]
archived_urls = [(e['target_url'], e['internet_archive_url']) for e in df.to_dict(orient='records')]
archived_urls = {e[0]:e[1] for e in archived_urls}

with open('archived_urls.json', 'w', encoding='utf-8') as jfile:
        json.dump(archived_urls, jfile, ensure_ascii=False)

In [ ]:
# all records urls
from collections import Counter

all_records_urls = []
all_records = 0
for file in tqdm(os.listdir('to_bibframe/manual')):
    df = pd.read_excel('to_bibframe/manual/' + file, sheet_name='Posts').fillna('')
    df = df[df['do PBL'] == True]
    all_records += len(df)
    urls = df['Link'].to_list()
    all_records_urls.extend(urls)

print('manual records:', len(all_records_urls))

for file in tqdm(os.listdir('to_bibframe/automatic_full')):
    with open('to_bibframe/automatic_full/' + file, 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)
        all_records += len(data)
        urls = [e['Link'] for e in data]
        all_records_urls.extend(urls)

for file in tqdm(os.listdir('to_bibframe/automatic_filtered')):
    with open('to_bibframe/automatic_filtered/' + file, 'r', encoding='utf-8') as jfile:
        data = json.load(jfile)
        all_records += len(data)
        urls = [e['Link'] for e in data]
        all_records_urls.extend(urls)

all_records_urls = [e for e in all_records_urls if not e.endswith('.jpg') and e]
print('all records urls:', len(all_records_urls))
print('all records urls unique:', len(set(all_records_urls)))

counter = Counter(all_records_urls)
duplicates = len([(e, counter[e]) for e in counter if counter[e] == 2])
print('all records urls - duplicates:', len(all_records_urls) - duplicates)

In [ ]:
# prepare urls for archiving
urls_to_archive = [e for e in list(all_records_urls) if e not in archived_urls]
with open('urls_to_archive.json', 'w', encoding='utf-8') as jfile:
    json.dump(urls_to_archive, jfile, ensure_ascii=False)


In [ ]:
# update archived_urls and urls_to_archive
with open('archived_urls.json', 'r', encoding='utf-8') as jfile:
    archived_urls = json.load(jfile)

to_update = {}
with open('results.jsonl', 'r', encoding='utf-8') as file:
    for line in file.readlines():
        line_dct = json.loads(line)
        if line_dct['status'] == 'success':
            to_update[line_dct['url']] = line_dct['archive_url']
archived_urls.update(to_update)

with open('archived_urls.json', 'w', encoding='utf-8') as jfile:
    json.dump(archived_urls, jfile, ensure_ascii=False)

In [ ]:
counter = 0
for key in all_records_urls:
    if key in archived_urls:
        counter += 1

print(counter)